# 2. Feature Engineering — создание признаков для прогнозирования РТО

**Цель этапа:**
- Создать набор признаков, улучшающих качество прогноза
- Реализовать временные лаги, скользящие средние и EWMA
- Добавить региональные агрегации для учёта географических паттернов
- Сформировать признаки взаимодействия между исходными переменными
- Подготовить данные для обучения моделей (split)


## 1. Импорт библиотек и настройка окружения

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Настройки отображения
pd.set_option('display.max_columns', None)

# Фиксация seed для воспроизводимости
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

## 2. Загрузка исходных данных

In [3]:
df = pd.read_csv('../../data/train.csv')
print(f"Датасет: {df.shape[0]} строк, {df.shape[1]} столбцов")

Датасет: 206150 строк, 19 столбцов



## 3. Базовые временные признаки

На основе результатов EDA установлено, что данные отсортированы по `new_id` и `Месяц`. Это позволяет корректно создавать лаговые признаки.

**Создаваемые признаки:**
- `rto_lag_1`, `rto_lag_2`, `rto_lag_3` — значения РТО за 1, 2 и 3 месяца до текущего
- `rto_lag_6` — значение РТО за полгода до текущего
- `rto_diff_1` — абсолютная разница между `rto_lag_1` и `rto_lag_2`
- `rto_pct_change` — процентное изменение между `rto_lag_1` и `rto_lag_2`

**Назначение:** лаги позволяют модели использовать историю продаж магазина; разностные признаки кодируют динамику изменений.

In [4]:
# Лаговые признаки
df['rto_lag_1'] = df.groupby('new_id')['РТО'].shift(1)
df['rto_lag_2'] = df.groupby('new_id')['РТО'].shift(2)
df['rto_lag_3'] = df.groupby('new_id')['РТО'].shift(3)
df['rto_lag_6'] = df.groupby('new_id')['РТО'].shift(6)

# Разностные признаки
df['rto_diff_1'] = df['rto_lag_1'] - df['rto_lag_2']
df['rto_pct_change'] = (df['rto_lag_1'] / (df['rto_lag_2'] + 1e-6)) - 1

df[['new_id', 'Месяц', 'РТО', 'rto_lag_1', 'rto_lag_2', 'rto_lag_6', 'rto_diff_1']].head()

,new_id,Месяц,РТО,rto_lag_1,rto_lag_2,rto_lag_6,rto_diff_1
0,0,1,22041969.33,NaN,NaN,NaN,NaN
1,0,2,23268490.57,22041969.33,NaN,NaN,NaN
2,0,3,24487732.21,23268490.57,22041969.33,NaN,1226521.24
3,0,4,23981980.29,24487732.21,23268490.57,NaN,1219241.64
4,0,5,26608343.78,23981980.29,24487732.21,NaN,-505751.92


Первые строки таблицы содержат `NaN` — это ожидаемое поведение, так как для первых месяцев каждого магазина отсутствуют предыдущие значения. Строки с `NaN` будут удалены на этапе формирования выборок (месяцы 1–6 не используются для обучения). Для прогнозируемого месяца 10 все лаговые признаки вычислены корректно.

## 4. Скользящие средние (MA и EWMA)

Скользящие средние сглаживают краткосрочные колебания и выделяют тренд.

**Простое скользящее среднее (MA):**

$$
MA_n = \frac{\sum_{i=t-n}^{t-1} \text{РТО}_i}{n}
$$

где $n$ — размер окна (3 или 6 месяцев). Берутся значения за предыдущие месяцы (без учёта текущего).

**Экспоненциальное скользящее среднее (EWMA):**

$$
EWMA_t = \alpha \cdot \text{РТО}_{t-1} + (1 - \alpha) \cdot EWMA_{t-1}
$$

где $\alpha = \frac{2}{\text{span} + 1}$ — коэффициент сглаживания. При span=3: $\alpha = 0.5$ (новые данные получают вес 50%). При span=6: $\alpha \approx 0.286$ (вес новых данных 28.6%).

EWMA придаёт больший вес более свежим наблюдениям, что особенно полезно для розничных данных, где актуальность недавних периодов выше.

**Создаваемые признаки:**
- `rto_ma_3`, `rto_ma_6` — простые скользящие средние за 3 и 6 месяцев
- `rto_ewm_3`, `rto_ewm_6` — экспоненциальные скользящие средние (span = 3 и 6)

Все признаки вычисляются с использованием `shift(1)`, чтобы избежать использования информации о текущем месяце (предотвращение data leakage).

In [5]:
# Простые скользящие средние (без учёта текущего месяца)
df['rto_ma_3'] = df.groupby('new_id')['РТО'].transform(
    lambda x: x.shift(1).rolling(3, min_periods=1).mean()
)
df['rto_ma_6'] = df.groupby('new_id')['РТО'].transform(
    lambda x: x.shift(1).rolling(6, min_periods=1).mean()
)

# Экспоненциальные скользящие средние
df['rto_ewm_3'] = df.groupby('new_id')['РТО'].transform(
    lambda x: x.shift(1).ewm(span=3, adjust=False).mean()
)
df['rto_ewm_6'] = df.groupby('new_id')['РТО'].transform(
    lambda x: x.shift(1).ewm(span=6, adjust=False).mean()
)


df[['new_id', 'Месяц', 'РТО', 'rto_ma_3', 'rto_ewm_3', 'rto_ma_6', 'rto_ewm_6']].head()

,new_id,Месяц,РТО,rto_ma_3,rto_ewm_3,rto_ma_6,rto_ewm_6
0,0,1,22041969.33,NaN,NaN,NaN,NaN
1,0,2,23268490.57,2.204197e+07,2.204197e+07,2.204197e+07,2.204197e+07
2,0,3,24487732.21,2.265523e+07,2.265523e+07,2.265523e+07,2.239240e+07
3,0,4,23981980.29,2.326606e+07,2.357148e+07,2.326606e+07,2.299107e+07
4,0,5,26608343.78,2.391273e+07,2.377673e+07,2.344504e+07,2.327419e+07



**Анализ:**

1. **MA3 vs MA6:** MA6 всегда ниже или равен MA3 (кроме месяца 2) — более длинное окно сильнее сглаживает колебания
2. **EWMA3 vs EWMA6:** EWMA3 быстрее реагирует на изменения (выше в месяц 4), EWMA6 стабильнее
3. **MA vs EWMA одинакового окна:** значения близки, но EWMA более чувствителен к тренду

**Значение для моделирования:**

- MA3 и MA6 дадут модели информацию о среднем уровне продаж за разные периоды
- EWMA3 и EWMA6 добавят информацию с учётом «старения» данных — свежие месяцы имеют больший вес
- В исходном эксперименте добавление EWMA улучшило скор с 95.34 до 95.44, что подтверждает полезность этого признака

## 5. Сезонность (циклическое кодирование месяца)

Месяц как категориальный признак (1–10) не передаёт модель цикличности: модель не знает, что январь (1) и декабрь (12) — соседние месяцы. Для этого используется циклическое кодирование через синус и косинус.

**Формулы:**

$$
\text{month\_sin} = \sin\left(\frac{2\pi \cdot \text{Месяц}}{12}\right)
$$

$$
\text{month\_cos} = \cos\left(\frac{2\pi \cdot \text{Месяц}}{12}\right)
$$

Значения синуса и косинуса образуют непрерывный цикл, где январь и декабрь оказываются рядом на единичной окружности.

**Создаваемые признаки:**
- `month_sin` — синус угла, соответствующего месяцу
- `month_cos` — косинус угла, соответствующего месяцу

In [6]:
# Циклическое кодирование месяца
df['month_sin'] = np.sin(2 * np.pi * df['Месяц'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['Месяц'] / 12)

df[['new_id', 'Месяц', 'month_sin', 'month_cos']].head()

,new_id,Месяц,month_sin,month_cos
0,0,1,0.500000,8.660254e-01
1,0,2,0.866025,5.000000e-01
2,0,3,1.000000,6.123234e-17
3,0,4,0.866025,-5.000000e-01
4,0,5,0.500000,-8.660254e-01


- Значения `month_cos` для месяца 3 близко к нулю (6.12e-17) — это вычислительная погрешность, фактически 0
- Переход от месяца 5 к 4 к 3 и далее показывает плавное изменение значений

В паре с категориальным признаком `Месяц` новые признаки позволяют модели улавливать как дискретные, так и непрерывные сезонные закономерности.

## 6. Региональные агрегации

Регион является важным фактором, влияющим на РТО: плотность населения, уровень конкуренции, покупательская способность различаются. Добавляем признаки, отражающие региональные паттерны.

**Создаваемые признаки:**

1. `shops_in_region` — количество магазинов в регионе (характеризует насыщенность и конкуренцию)

2. `region_avg_rto_3` — средний РТО по региону за последние 3 месяца (с лагом 1, без учёта текущего месяца)

3. `rto_vs_region` — отношение РТО магазина к среднему по региону (показывает, насколько магазин эффективнее/хуже среднего)

Эти признаки помогут модели учитывать региональную специфику и сравнивать магазины внутри одного региона.

In [7]:
# Количество магазинов в регионе
df['shops_in_region'] = df.groupby('Регион')['new_id'].transform('nunique')

# Средний РТО по региону за последние 3 месяца
df['region_avg_rto_3'] = df.groupby(['Регион', 'Месяц'])['РТО'].transform('mean')
df['region_avg_rto_3'] = df.groupby('Регион')['region_avg_rto_3'].shift(1).rolling(3, min_periods=1).mean()

# Отношение РТО магазина к среднему по региону
df['rto_vs_region'] = df['РТО'] / (df['region_avg_rto_3'] + 1e-6)

# Вывод 
df[['new_id', 'Регион', 'Месяц', 'РТО', 'shops_in_region', 'region_avg_rto_3', 'rto_vs_region']].head()

,new_id,Регион,Месяц,РТО,shops_in_region,region_avg_rto_3,rto_vs_region
0,0,Краснодарский край,1,22041969.33,1039,NaN,NaN
1,0,Краснодарский край,2,23268490.57,1039,2.365441e+07,0.983685
2,0,Краснодарский край,3,24487732.21,1039,2.403359e+07,1.018896
3,0,Краснодарский край,4,23981980.29,1039,2.512568e+07,0.954481
4,0,Краснодарский край,5,26608343.78,1039,2.632207e+07,1.010876


## 7. Признаки взаимодействия

Признаки взаимодействия создаются путём комбинирования исходных переменных. Они позволяют модели улавливать нелинейные зависимости, которые сложно выразить через отдельные признаки.

**Создаваемые признаки:**

1. **`traffic_per_capita`** — пешеходный и автомобильный трафик на душу населения

   $$
   \text{traffic\_per\_capita} = \frac{\text{Трафик пеший} + \text{Трафик авто}}{\text{Численность населения} + \varepsilon}
   $$

   - в районах с высокой плотностью населения и большим трафиком ожидается более высокий РТО.

2. **`rto_per_cash`** — отношение РТО прошлого месяца к количеству касс

   $$
   \text{rto\_per\_cash} = \frac{\text{rto\_lag\_1}}{\text{Количество касс} + \varepsilon}
   $$

   - эффективность работы одной кассы; высокое значение может указывать на загруженность магазина.

3. **`population_density`** — плотность населения

   $$
   \text{population\_density} = \frac{\text{Численность населения}}{\text{Количество домохозяйств} + \varepsilon}
   $$

   - характеризует тип района (плотная городская застройка vs частный сектор).

Константа $\varepsilon = 10^{-6}$ добавлена для избежания деления на ноль.

In [8]:
EPS = 1e-6

# Признаки взаимодействия
df['traffic_per_capita'] = (df['Трафик пеший, в час'] + df['Трафик авто, в час']) / (df['Численность населения'] + EPS)
df['rto_per_cash'] = df['rto_lag_1'] / (df['Количество касс'] + EPS)
df['population_density'] = df['Численность населения'] / (df['Количество домохозяйств'] + EPS)

df[['new_id', 'Месяц', 'traffic_per_capita', 'rto_per_cash', 'population_density']].head()

,new_id,Месяц,traffic_per_capita,rto_per_cash,population_density
0,0,1,0.02451,NaN,19.137725
1,0,2,0.02451,2.204197e+06,19.137725
2,0,3,0.02451,2.326849e+06,19.137725
3,0,4,0.02451,2.448773e+06,19.137725
4,0,5,0.02451,2.398198e+06,19.137725


## 8. Формирование обучающей и валидационной выборок

После создания всех признаков необходимо:
1. Удалить строки с `NaN` (первые месяцы каждого магазина, где нет истории для лагов)
2. Разделить данные на обучающую (месяцы 7–9) и валидационную (месяц 10) выборки
3. Применить логарифмическую трансформацию к целевой переменной

- распределение РТО скошено вправо (выявлено в EDA). Логарифмирование приближает распределение к нормальному, что улучшает сходимость градиентных методов в деревьях.

In [9]:
# Удаление строк с пропусками (NaN)
df_model = df.dropna().copy()
print(f"Размер после удаления NaN: {len(df_model)} строк")

# Разделение на train (месяцы 7-9) и validation (месяц 10)
train_df = df_model[df_model['Месяц'] <= 9].copy()
val_df = df_model[df_model['Месяц'] == 10].copy()

print(f"Обучающая выборка: {len(train_df)} строк (месяцы 7-9)")
print(f"Валидационная выборка: {len(val_df)} строк (месяц 10)")

# Логарифмическая трансформация целевой переменной
y_train_log = np.log1p(train_df['РТО'].values)
y_val_log = np.log1p(val_df['РТО'].values)

print(f"Размер y_train_log: {len(y_train_log)}")
print(f"Размер y_val_log: {len(y_val_log)}")

Размер после удаления NaN: 82460 строк
Обучающая выборка: 61845 строк (месяцы 7-9)
Валидационная выборка: 20615 строк (месяц 10)
Размер y_train_log: 61845
Размер y_val_log: 20615



**Анализ:**

- Все магазины вошли в валидационную выборку (прогноз на месяц 10)
- Обучающая выборка содержит 3 месяца на магазин (7, 8, 9)
- Месяцы 1–6 исключены из-за отсутствия полной истории для лагов

**Значение для моделирования:**

Выборки готовы к обучению. Размер данных достаточен для обучения ансамбля моделей (CatBoost, LightGBM, XGBoost). 


## 9. Итоги: список созданных признаков

In [10]:
# Список всех признаков для моделирования
feature_cols = [
    'Месяц', 'month_sin', 'month_cos',
    'shops_in_region', 'region_avg_rto_3', 'rto_vs_region',
    'rto_lag_1', 'rto_lag_2', 'rto_lag_3', 'rto_lag_6',
    'rto_ma_3', 'rto_ma_6', 'rto_ewm_3', 'rto_ewm_6',
    'rto_diff_1', 'rto_pct_change',
    'traffic_per_capita', 'rto_per_cash', 'population_density',
    'Численность населения', 'Количество домохозяйств',
    'Трафик пеший, в час', 'Количество касс', 'Флаг алкогольной лицензии',
    'Школы (300 м)', 'Остановки (300 м)', 'Продуктовые магазины (500 м)',
    'Пятерочки (500 м)'
]

cat_features = ['Дата открытия, категориальный', 'Регион']

print(f"Всего числовых признаков: {len(feature_cols)}")
print(f"Категориальных признаков: {len(cat_features)}")
print(f"Общее количество признаков: {len(feature_cols) + len(cat_features)}")

Всего числовых признаков: 28
Категориальных признаков: 2
Общее количество признаков: 30


In [11]:



print(f"Числовые признаки ({len(feature_cols)}):")
for i, f in enumerate(feature_cols, 1):
    print(f"  {i:2d}. {f}")

print(f"\nКатегориальные признаки ({len(cat_features)}):")
for i, f in enumerate(cat_features, 1):
    print(f"  {i:2d}. {f}")

print(f"\nВсего признаков: {len(feature_cols) + len(cat_features)}")

Числовые признаки (28):
   1. Месяц
   2. month_sin
   3. month_cos
   4. shops_in_region
   5. region_avg_rto_3
   6. rto_vs_region
   7. rto_lag_1
   8. rto_lag_2
   9. rto_lag_3
  10. rto_lag_6
  11. rto_ma_3
  12. rto_ma_6
  13. rto_ewm_3
  14. rto_ewm_6
  15. rto_diff_1
  16. rto_pct_change
  17. traffic_per_capita
  18. rto_per_cash
  19. population_density
  20. Численность населения
  21. Количество домохозяйств
  22. Трафик пеший, в час
  23. Количество касс
  24. Флаг алкогольной лицензии
  25. Школы (300 м)
  26. Остановки (300 м)
  27. Продуктовые магазины (500 м)
  28. Пятерочки (500 м)

Категориальные признаки (2):
   1. Дата открытия, категориальный
   2. Регион

Всего признаков: 30


## 10. Сохранение данных для работы с моделями



In [12]:
import joblib
import os

# Создаём папку processed
os.makedirs('../../data/processed', exist_ok=True)

# Формируем матрицы признаков
X_train = train_df[feature_cols + cat_features].copy()
X_val = val_df[feature_cols + cat_features].copy()

# Сохраняем всё в один файл
joblib.dump((X_train, X_val, y_train_log, y_val_log), '../../data/processed/model_data.pkl')

# Сохраняем список категориальных признаков отдельно
joblib.dump(cat_features, '../../data/processed/cat_features.pkl')

print("Сохраненные данные")
print(f"  X_train: {X_train.shape}")
print(f"  X_val: {X_val.shape}")
print(f"  y_train_log: {len(y_train_log)}")
print(f"  y_val_log: {len(y_val_log)}")
print(f"  cat_features: {cat_features}")

Сохраненные данные
  X_train: (61845, 30)
  X_val: (20615, 30)
  y_train_log: 61845
  y_val_log: 20615
  cat_features: ['Дата открытия, категориальный', 'Регион']
